In [1]:
import json
import numpy as np
from pathlib import Path
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers import Tokenizer, AddedToken
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from huggingface_hub import hf_hub_download

In [2]:
path = hf_hub_download("roneneldan/TinyStories",
                       "TinyStoriesV2-GPT4-train.txt",
                       repo_type="dataset")

In [2]:
tokenizer_path = Path("./tokenizer/tokenizer.json")

In [3]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))

In [4]:
chunk_chars=1 << 24
sep="<|endoftext|>"

In [34]:
def iter_documents(path, sep="<|endoftext|>", chunk_chars=1 << 24):
    buf = ""
    with open(path, "r", encoding="utf-8") as f:
        while True:
            chunk = f.read(chunk_chars)
            if not chunk:
                break
            buf += chunk
            parts = buf.split(sep)
            buf = parts.pop()          # tail may be a partial document
            for p in parts:
                p = p.strip()
                if p:
                    yield p
    buf = buf.strip()
    if buf:
        yield buf

In [35]:
def iter_batches(path, batch_size=1000):
    batch = []
    for doc in iter_documents(path):
        batch.append(doc)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [40]:
EOS = tokenizer.token_to_id(sep)   # 1

def write_bin(path_in, path_out, batch_size=10_000):
    n_docs = n_tokens = 0
    with open(path_out, "wb") as f:
        for batch in iter_batches(path_in, batch_size):
            ids = []
            for enc in tokenizer.encode_batch_fast(batch):
                ids.extend(enc.ids)
                ids.append(EOS)
            np.asarray(ids, dtype=np.uint16).tofile(f)#tofile on an open handle appends,
            n_docs += len(batch)
            n_tokens += len(ids)
    return n_docs, n_tokens

In [ ]:
%%time

path_out = Path("../data/tokenized_train.bin")
write_bin(path, path_out, batch_size=10_000)

CPU times: user 1h 1min 4s, sys: 14.2 s, total: 1h 1min 18s
Wall time: 3min 17s


(2717495, 533759198)

In [9]:
data = np.memmap(path_out, dtype=np.uint16, mode="r")

In [11]:
type(data), data.shape, data[:10], data[-10:]

(numpy.memmap,
 (533759198,),
 memmap([439, 448, 261, 405, 409, 285, 261, 399, 506, 410], dtype=uint16),
 memmap([2417,  484,   16,  343,  328,   14,  321,   59,   81,    1],
        dtype=uint16))

In [45]:
path_valid = hf_hub_download("roneneldan/TinyStories",
                       "TinyStoriesV2-GPT4-valid.txt",
                       repo_type="dataset")

In [ ]:
%%time

path_out = Path("../data/tokenized_valid.bin")
write_bin(path_valid, path_out, batch_size=10_000)

CPU times: user 33.4 s, sys: 134 ms, total: 33.6 s
Wall time: 1.78 s


(27630, 5390305)